# H5N1 Pipeline

Author: Alexander Maksiaev

Purpose: Scrape files from EpiFlu for avian flu project

In [1]:
# Have user type in username and password

username = input("Username: ")
password = input("Password: ")
browser = input("Browser: ")
sleep_time = input("Seconds to sleep in between clicks: ")
start_date = input("Start date (format: YYYY-MM-DD): ")
end_date = input("End date (format: YYYY-MM-DD): ")
# hashed_password = hashlib.sha256(password.encode()).hexdigest()

In [ ]:
# Opening the website

# import requests
# from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.relative_locator import locate_with
from selenium.webdriver.common.alert import Alert 

import time 

def open_gisaid(browser, username, password, sleep_time, start_date, end_date):

    if browser=="Firefox":
        # If you want to open Firefox
        driver = webdriver.Firefox()
    elif browser=="Chrome": # if Chrome...
        driver = webdriver.Chrome()
    else: # Edge, probably
        driver = webdriver.Edge()

    # How many seconds should pass between tasks
    sleep_time = int(sleep_time)

    # Requested URL
    driver.get("https://www.epicov.org/epi3/frontend#")

    # Wait for it to load, otherwise it won't work
    # driver.implicitly_wait(20)
    # username_field = wait.until(EC.element_to_be_clickable((By.NAME, 'login')))
    # password_field = wait.until(EC.element_to_be_clickable((By.NAME, 'password')))

    time.sleep(sleep_time)

    # Input username and password
    username_field = driver.find_element(By.NAME, "login")
    password_field = driver.find_element(By.NAME, "password")
    submit_button = driver.find_element(By.CLASS_NAME, "form_button_submit")
    username_field.send_keys(username)
    password_field.send_keys(password)


    # Wait for it to load
    time.sleep(sleep_time)
    # submit_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'form_button_submit')))
    submit_button.click()

    # Wait for it to load again
    time.sleep(sleep_time)

    # Find the correct tab
    epiflu_link = driver.find_element(By.XPATH, "//*[contains(text(), 'EpiFlu™')]")
    # epiflu_link = wait.until(EC.visibility_of_element_located((By.XPATH, "//*[contains(text(), 'EpiFlu™')]")))
    epiflu_link.click()

    time.sleep(sleep_time)

    # Search tab 
    search_link = driver.find_element(By.XPATH, "//*[contains(text(), 'Search')]")
    search_link.click()

    time.sleep(sleep_time)

    # Type: A
    # Gives "internal server error" without action chains
    type_a = driver.find_element(By.XPATH, "//*[contains(@class, 'sys-form-filine-td')]//option[@value='A']")
    ActionChains(driver).move_to_element(type_a).pause(1).click(type_a).perform() 

    time.sleep(sleep_time)

    # H: 5
    h_5 = driver.find_element(By.XPATH, "//*[contains(@class, 'sys-form-filine-td')]//option[@value='5']") # The second multi-select
    ActionChains(driver).move_to_element(h_5).pause(1).click(h_5).perform() 

    time.sleep(sleep_time)

    # Find the ID for N

    # N: 1
    n_1 = driver.find_element(By.XPATH, "//*[contains(@class, 'sys-form-filine-td')][3]//option[@value='1']") # The third multi-select
    ActionChains(driver).move_to_element(n_1).pause(1).click(n_1).perform() 

    time.sleep(sleep_time)
    
    # Location: North America
    location_northa = driver.find_element(By.XPATH, "//*[contains(@class, 'sys-event-hook sys-fi-mark')]//option[@value='6440']")
    # location_northa = location_northa_1.location_once_scrolled_into_view
    driver.execute_script("arguments[0].scrollIntoView();", location_northa)
    ActionChains(driver).move_to_element(location_northa).pause(1).click(location_northa).perform()
    # location_northa.click()

    time.sleep(sleep_time)

    # Segments: PB2, PB1, PA, HA, NP, NA, MP, NS (all EXCEPT HE, P3)
    segment_list = ['PB2', 'PB1', 'PA', 'HA', 'NP', 'NA', 'MP', 'NS']
    for segment in segment_list:
        xpath = "//*[contains(@class, 'sys-form-fi-cb sys-fi-mark')]//input[@value='" + segment + "']"
        pb2 = driver.find_element(By.XPATH, xpath)
        ActionChains(driver).move_to_element(pb2).pause(1).click(pb2).perform()

    # Start submission: 2023-03-18
    # End submission: 2025-03-31

    # Insert dates
    first_date = driver.find_element(By.XPATH, "//*[contains(text(),'Submission date from')]/ancestor::td/following-sibling::td//input[@class='sys-event-hook sys-fi-mark hasDatepicker']") # First date
    first_date.send_keys(start_date)

    last_date = driver.find_element(By.XPATH, "//*[contains(text(),'Submission date from')]/ancestor::td/following-sibling::td[3]//input[@class='sys-event-hook sys-fi-mark hasDatepicker']") # Second date
    last_date.send_keys(end_date)

    time.sleep(sleep_time)

    # Split search into batches by date if >= 10k sequences

    # Search

    search_button = driver.find_element(By.XPATH, "//*[contains(@class, 'buttons container-slot')]//button[@accesskey='g']")
    search_button.click()

    # Choose all files -- unless the number of sequences is >10k, then do it in batches
    time.sleep(sleep_time)

    # # Double the sleep time to load
    # time.sleep(sleep_time)

    # Select all (at first)
    select_checkbox = driver.find_element(By.XPATH, "//*[contains(@class, 'yui-dt-first yui-dt-last')]//input[@type='checkbox']")
    select_checkbox.click()

    # Press the select button and select up to 10,000 entries. If there's more afterwards, download the first 10k and continue until we've reached the end.

    # Press the select button
    select_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Select')]")
    ActionChains(driver).move_to_element(select_button).pause(1).click(select_button).perform()    

    time.sleep(sleep_time)

    # Get all the entries
    iframe = driver.find_element(By.NAME, "wjob")
    driver.switch_to.frame(iframe)
    entry_text = driver.find_element(By.XPATH, "//textarea")
    full_text = entry_text.text
    entries = re.split(r"[,\n]", full_text)

    # Get rid of empty strings
    for entry in entries:
        if entry == '':
            entries.remove(entry)

    # While there are >10k sequences, add each set of 10k to a list
    # 1 string ~= 20 sequences
    thresh = 10000//20
    ten_thousands = [entries[:thresh]]
    entries = entries[thresh:]

    while len(entries) > thresh: 
        ten_k = entries[:thresh]
        ten_thousands.append(ten_k)
        entries = entries[thresh:]
        print(len(entries))

    print(ten_thousands)

    # Go back and select the nth 10k from the entries frame


    # Download metadata as XLS

    # Download segment sequences as DNA FASTA file

In [11]:
open_gisaid(browser, username, password, sleep_time, start_date, end_date)

129
79
29
[['EPI_ISL_19785793', ' EPI_ISL_19785995-19786015', ' EPI_ISL_19787948', 'EPI_ISL_19787950-19787999', ' EPI_ISL_19788001-19788004', ' EPI_ISL_19788006-19788013', 'EPI_ISL_19792227-19792253', ' EPI_ISL_19792255-19792256', ' EPI_ISL_19792258-19792270', 'EPI_ISL_19792272-19792279', ' EPI_ISL_19792283-19792287', ' EPI_ISL_19792289-19792292', 'EPI_ISL_19792294-19792297', ' EPI_ISL_19792299-19792301', ' EPI_ISL_19792303-19792306', 'EPI_ISL_19792308-19792311', ' EPI_ISL_19792313-19792319', ' EPI_ISL_19792321-19792322', 'EPI_ISL_19792324-19792325', ' EPI_ISL_19792327', ' EPI_ISL_19792329-19792331', 'EPI_ISL_19792333-19792339', ' EPI_ISL_19792341-19792349', ' EPI_ISL_19792351-19792352', 'EPI_ISL_19792354-19792364', ' EPI_ISL_19792366-19792367', ' EPI_ISL_19792369-19792380', 'EPI_ISL_19792385-19792388', ' EPI_ISL_19792390-19792392', ' EPI_ISL_19792394', 'EPI_ISL_19792396-19792397', ' EPI_ISL_19792399-19792404', ' EPI_ISL_19792406-19792407', 'EPI_ISL_19792410-19792432', ' EPI_ISL_197924

In [ ]:
# # Search and download relevant data



# driver.switch_to.frame("frameName.0.child")

# driver.implicitly_wait(10)

# epipox_tab = driver.find_element(By.LINK_TEXT, "EpiPox™")

# for tab in range(4):
#     main_nav.send_keys(Keys.TAB)

# main_nav.send_keys(Keys.ENTER)

# time.sleep(2)

# epipox_tab.click()

# # function onclick(event) {
# #   sys.call('c_st4hhy_2b1', 'Go', new Object({
# #     'page': 'mpox'
# #   }));
# # }